Bag of words y TF-IDF

In [1]:
import json
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

textos = []
# Ajusta la ruta dependiendo de dónde ejecutes el cuaderno
with open('../data/processed/congreso_limpio.jsonl', 'r', encoding='utf-8') as f:
    for linea in f:
        dato = json.loads(linea)
        # Si el JSONL guarda el texto como lista de tokens, usa: " ".join(dato['tokens'])
        # Si lo guarda como texto plano, usa la clave correspondiente (ej. 'texto')
        texto_procesado = dato.get('texto', '') 
        textos.append(texto_procesado)

vectorizador_bow = CountVectorizer()
matriz_bow = vectorizador_bow.fit_transform(textos)

vectorizador_tfidf = TfidfVectorizer()
matriz_tfidf = vectorizador_tfidf.fit_transform(textos)

print(f"Palabras únicas en el vocabulario: {len(vectorizador_bow.get_feature_names_out())}")
print(f"Dimensiones de la matriz BoW: {matriz_bow.shape}")

Palabras únicas en el vocabulario: 55346
Dimensiones de la matriz BoW: (2468, 55346)


Palabras que no aparecen

In [3]:
import os
from gensim.models import KeyedVectors
from huggingface_hub import hf_hub_download

ruta_local = "../models/eswiki_20180420_300d.txt"

if os.path.exists(ruta_local):
    print("Cargando modelo local...")
    modelo_w2v = KeyedVectors.load_word2vec_format(ruta_local)
else:
    print("Modelo local no encontrado. Descargando desde Hugging Face...")
    ruta_descarga = hf_hub_download(
        repo_id="Word2vec/wikipedia2vec_eswiki_20180420_300d",
        filename="eswiki_20180420_300d.txt"
    )
    modelo_w2v = KeyedVectors.load_word2vec_format(ruta_descarga)

print("¡Modelo cargado!")

Cargando modelo local...
¡Modelo cargado!


In [4]:
vocabulario_corpus = vectorizador_bow.get_feature_names_out()
palabras_oov = [palabra for palabra in vocabulario_corpus if palabra not in modelo_w2v.key_to_index]
porcentaje_oov = (len(palabras_oov) / len(vocabulario_corpus)) * 100

print(f"Palabras fuera del vocabulario (OOV): {len(palabras_oov)}")
print(f"Porcentaje de pérdida: {porcentaje_oov:.2f}%")

Palabras fuera del vocabulario (OOV): 13602
Porcentaje de pérdida: 24.58%


In [5]:
import numpy as np

# 1. Muestra de palabras que faltan
print("Ejemplos OOV:", palabras_oov[:40])

# 2. Cobertura real: % de palabras de los textos (no del vocabulario) que sí están en el modelo
frecuencias = np.asarray(matriz_bow.sum(axis=0)).ravel()
vocab = list(vocabulario_corpus)
total = frecuencias.sum()
cubiertas = sum(f for p, f in zip(vocab, frecuencias) if p in modelo_w2v.key_to_index)
print(f"Cobertura real de tokens: {cubiertas / total * 100:.2f}%")

Ejemplos OOV: ['1978ko', '19aren', '2009an', '20102022', '2010ean', '20115', '2018an', '2019an', '2022an', '2023an', '2023ko', '20242026', '2024an', '2024koaren', '2025ean', '2025eko', '2025era', '2026an', '23155', '25ean', '25n', '25è', '28an', '28tik', '31bis', '32rekin', '39420', '40eko', '48bis', '5ak', '60koa', '9233', '9526', 'aaay', 'abaixa', 'abaixar', 'abaixat', 'abanderats', 'abanderen', 'abandonades']
Cobertura real de tokens: 97.64%


Un vector por texto

In [6]:
import numpy as np

analizador = vectorizador_bow.build_analyzer()

def vector_del_texto(texto):
    palabras = [p for p in analizador(texto) if p in modelo_w2v.key_to_index]
    if not palabras:
        return np.zeros(modelo_w2v.vector_size)
    return np.mean([modelo_w2v[p] for p in palabras], axis=0)

matriz_w2v = np.array([vector_del_texto(t) for t in textos])
sin_vector = int((matriz_w2v == 0).all(axis=1).sum())

print("Dimensiones:", matriz_w2v.shape)
print("Textos sin ninguna palabra con vector:", sin_vector)

Dimensiones: (2468, 300)
Textos sin ninguna palabra con vector: 0


In [7]:
import json
import os
from scipy import sparse

# 1. Archivo principal: tus textos originales + el vector de 300 números
with open('../data/processed/congreso_limpio.jsonl', 'r', encoding='utf-8') as f:
    lineas = [json.loads(l) for l in f]

assert len(lineas) == matriz_w2v.shape[0], "El número de textos no coincide con la matriz"

with open('../data/processed/congreso_vectorizado.jsonl', 'w', encoding='utf-8') as f_out:
    for dato, vector in zip(lineas, matriz_w2v):
        dato['vector_w2v'] = [round(float(x), 5) for x in vector]
        f_out.write(json.dumps(dato, ensure_ascii=False) + '\n')

print(f"Éxito: {len(lineas)} textos exportados con sus 300 coordenadas.")

# 2. Las otras dos representaciones y el vocabulario
carpeta = "../data/representations/congreso"
os.makedirs(carpeta, exist_ok=True)

sparse.save_npz(f"{carpeta}/bow.npz", matriz_bow)
sparse.save_npz(f"{carpeta}/tfidf.npz", matriz_tfidf)
with open(f"{carpeta}/vocabulario.json", "w", encoding="utf-8") as f:
    json.dump(vectorizador_bow.get_feature_names_out().tolist(), f, ensure_ascii=False)

print("Matrices y vocabulario guardados en:", carpeta)

Éxito: 2468 textos exportados con sus 300 coordenadas.
Matrices y vocabulario guardados en: ../data/representations/congreso


In [8]:
import json
from scipy import sparse

carpeta = "../data/representations/congreso"

# 1. El archivo principal
with open('../data/processed/congreso_vectorizado.jsonl', encoding='utf-8') as f:
    primera = json.loads(f.readline())
print("Texto:", primera['texto'])
print("Números del vector:", len(primera['vector_w2v']), "->", primera['vector_w2v'][:5])

# 4. El vocabulario
with open(f"{carpeta}/vocabulario.json", encoding="utf-8") as f:
    vocabulario = json.load(f)
print("\nPalabras:", len(vocabulario), "->", vocabulario[:8], "...", vocabulario[-3:])

# 2 y 3. Las dos matrices, fila 0, con el nombre de cada palabra
for nombre in ["bow", "tfidf"]:
    matriz = sparse.load_npz(f"{carpeta}/{nombre}.npz")
    fila = matriz[0]
    print(f"\n{nombre} - fila 0:")
    for columna, valor in zip(fila.indices, fila.data):
        print("  ", vocabulario[columna], "=", round(float(valor), 3))

Texto: Sí, prometo.
Números del vector: 300 -> [0.1278, -0.2187, 0.08855, 0.30435, 0.18515]

Palabras: 55346 -> ['00', '000', '0000', '0005', '005', '008', '01', '010'] ... ['útil', 'útiles', 'útils']

bow - fila 0:
   sí = 1.0
   prometo = 1.0

tfidf - fila 0:
   sí = 0.299
   prometo = 0.954
